# Analyzing an Author's Corpus with RAG

This notebook is the interactive front end for source-agnostic ingestion, exact catalog queries, and semantic retrieval. Reusable implementation lives in the `author_corpus` package.

## Local configuration and privacy

Copy `corpus.example.yaml` to the ignored `corpus.local.yaml`, configure private input paths there, and set `AUTHOR_CORPUS_CONFIG`. Select the `Python 3.14 (author-corpus-rag)` kernel before running the notebook. Committed notebook outputs are stripped automatically.

In [ ]:
import os
from pathlib import Path

from dotenv import load_dotenv

import author_corpus
from author_corpus import (
    INDEX_PIPELINE_VERSION,
    AuthorIdentity,
    AuthorScope,
    CacheLayout,
    CorpusCatalog,
    NotebookRunOptions,
    TimingLog,
    build_catalog_tools,
    corpus_fingerprint,
    load_corpus_config,
    parse_author_aliases,
    resolve_author_query,
)
from author_corpus.persistence import write_manifest

timings = TimingLog()
project_root = Path(author_corpus.__file__).resolve().parents[2]
load_dotenv(project_root / ".env")
run_options = NotebookRunOptions.from_environment(os.environ)
print("allow missing index build:", run_options.allow_index_build)
print("run grounded answer:", run_options.run_grounded_answer)

config_value = os.getenv("AUTHOR_CORPUS_CONFIG")
if not config_value:
    raise RuntimeError("Set AUTHOR_CORPUS_CONFIG to an ignored corpus.local.yaml file.")

config_path = Path(config_value).expanduser()
if not config_path.is_absolute():
    config_path = project_root / config_path
config_path = config_path.resolve()

## Load and validate logical documents

Missing metadata is reported explicitly but does not prevent semantic indexing. Blocking extraction problems fail before indexes are built.

In [ ]:
load_started = timings.start()
load_result = load_corpus_config(config_path)

print("corpus:", load_result.name or config_path.stem)
print("documents:", len(load_result.documents))
print("warnings:", len(load_result.warnings))
print("errors:", len(load_result.errors))

for issue in load_result.issues:
    print(f"{issue.severity.upper():7} {issue.code}: {issue.path or '<corpus>'} — {issue.message}")

load_result.raise_for_errors()
documents = load_result.documents
print(timings.finish("Load and validate corpus", load_started))

## Build the exact catalog

Counts and exhaustive metadata questions belong here, not in top-k vector retrieval.

In [ ]:
catalog_started = timings.start()
index_options = {
    "chunk_size": 1024,
    "chunk_overlap": 200,
    "index_pipeline_version": INDEX_PIPELINE_VERSION,
    "embedding_model": os.getenv("EMBEDDING_MODEL", "BAAI/bge-small-en-v1.5"),
}
fingerprint = corpus_fingerprint(documents, options=index_options)
layout = CacheLayout(project_root / ".cache", fingerprint)
layout.ensure()

catalog = CorpusCatalog(layout.catalog_path)
catalog.rebuild(documents)
write_manifest(layout, documents, options=index_options)

default_author = os.getenv("AUTHOR_CORPUS_DEFAULT_AUTHOR")
author_identity = None
default_scope = AuthorScope()
if default_author:
    author_identity = AuthorIdentity.from_catalog(
        default_author=default_author,
        aliases=parse_author_aliases(os.getenv("AUTHOR_CORPUS_DEFAULT_AUTHOR_ALIASES")),
        catalog_authors=(name for name, _ in catalog.author_counts()),
    )
    default_scope = AuthorScope.for_author(author_identity.canonical_name)

catalog_stats = catalog.stats()
print(timings.finish("Build exact catalog", catalog_started))
catalog_stats

In [ ]:
catalog.author_counts()

In [ ]:
catalog_tools = build_catalog_tools(catalog)
[tool.metadata.name for tool in catalog_tools]

## Build or load retrieval profiles

The vector and BM25 indexes are stored under the corpus fingerprint, so changed documents or chunking settings produce new cache entries. Loading existing indexes is safe during `Run All`; building a missing vector index requires `AUTHOR_CORPUS_ALLOW_INDEX_BUILD=1`.

In [ ]:
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

from author_corpus.indexing import load_or_build_vector_index, load_vector_index, vector_index_exists

vector_started = timings.start()
embed_model = HuggingFaceEmbedding(
    model_name=index_options["embedding_model"],
    device="cpu",
)
if vector_index_exists(layout):
    vector_index = load_vector_index(layout, embed_model=embed_model)
    loaded_from_cache = True
elif run_options.allow_index_build:
    vector_index, loaded_from_cache = load_or_build_vector_index(
        documents,
        layout,
        embed_model=embed_model,
        chunk_size=index_options["chunk_size"],
        chunk_overlap=index_options["chunk_overlap"],
    )
else:
    raise RuntimeError(
        "The vector index is missing. Set AUTHOR_CORPUS_ALLOW_INDEX_BUILD=1, restart the kernel, and rerun this cell."
    )
print("loaded from cache:", loaded_from_cache)
print(timings.finish("Load or build vector index", vector_started))

## Route a question and inspect semantic evidence

The deterministic router sends exact metadata questions to the SQLite catalog, broad exploration to diverse dense discovery, and focused questions to dense plus BM25 evidence. It reports the rule and matched text instead of inventing a confidence score. Semantic retrieval remains intentionally non-exhaustive.

In [ ]:
from author_corpus import build_retrieval_profiles

profiles_started = timings.start()
retrieval_profiles, lexical_loaded_from_cache = build_retrieval_profiles(
    vector_index,
    layout,
    candidate_pool_size=30,
    evidence_passages_per_document=3,
)
discovery_search = retrieval_profiles.discovery
evidence_search = retrieval_profiles.hybrid_evidence
print("lexical index loaded from cache:", lexical_loaded_from_cache)
print(timings.finish("Load or build retrieval profiles", profiles_started))

In [ ]:
from IPython.display import Markdown, display

from author_corpus import QueryRoute, execute_catalog_query, route_query

semantic_query = "What practical advice does the author give readers?"
minimum_document_author_fraction = 0.8
routing_started = timings.start()
route_decision = route_query(semantic_query)
routing_timing = timings.finish("Query routing", routing_started)

print("query:", route_decision.query)
print("route:", route_decision.route)
print("rule:", route_decision.rule_id)
print("matched signals:", route_decision.matched_signals or "<focused default>")
print("rationale:", route_decision.rationale)
print(routing_timing)

selected_search = None
semantic_result = None
exact_result = None
if route_decision.route is QueryRoute.EXACT_CATALOG:
    print("catalog tool:", route_decision.catalog_tool)
    exact_started = timings.start()
    exact_result = execute_catalog_query(
        catalog,
        route_decision,
        default_author=os.getenv("AUTHOR_CORPUS_DEFAULT_AUTHOR"),
    )
    exact_timing = timings.finish("Exact catalog execution", exact_started)
    display(Markdown(exact_result.to_markdown()))
    print("resolved arguments:", exact_result.arguments.model_dump())
    print(exact_timing)
    print("No semantic search or model generation was run.")
else:
    selected_search = discovery_search if route_decision.route is QueryRoute.BROAD_DISCOVERY else evidence_search
    identity_started = timings.start()
    author_resolution = resolve_author_query(semantic_query, author_identity)
    print("retrieval query:", author_resolution.retrieval_query)
    print("author references:", author_resolution.matched_references or "<none>")
    print(timings.finish("Author identity resolution", identity_started))
    search_started = timings.start()
    semantic_result = selected_search.search(
        author_resolution.retrieval_query,
        minimum_document_author_fraction=minimum_document_author_fraction,
    )
    search_timing = timings.finish("Routed semantic retrieval", search_started)
    print("strategy:", semantic_result.strategy)
    print("score kind:", semantic_result.score_kind)
    print("exhaustive:", semantic_result.exhaustive)
    print("candidates inspected:", semantic_result.inspected_candidates)

    for passage in semantic_result.passages:
        score = "n/a" if passage.score is None else f"{passage.score:.4f}"
        print(f"\n{passage.rank}. {passage.title} — score {score}")
        contributions = ", ".join(
            f"{item.method} rank {item.rank} ({item.score_kind}={item.score:.4f})"
            for item in passage.retrieval_contributions
            if item.score is not None
        )
        print("   retrieval:", contributions or "<not recorded>")
        print("   document:", passage.document_id)
        print("   source:", passage.canonical_source_uri or "<no canonical source>")
        print("   section:", " > ".join(passage.section_path) or "<document introduction>")
        author_fraction = passage.document_author_fraction
        author_percent = "unknown" if author_fraction is None else f"{author_fraction:.1%}"
        quoted_fraction = passage.quoted_speech_fraction
        quoted_percent = "unknown" if quoted_fraction is None else f"{quoted_fraction:.1%}"
        uncertain_fraction = passage.uncertain_voice_fraction
        uncertain_percent = "unknown" if uncertain_fraction is None else f"{uncertain_fraction:.1%}"
        print(
            f"   voice: {passage.passage_voice}; author: {author_percent}; quoted: {quoted_percent}; "
            f"uncertain: {uncertain_percent}"
        )
        if passage.attributed_speakers:
            print("   attributed speakers:", ", ".join(passage.attributed_speakers))
        print("   passage:", passage.text[:500].replace("\n", " "))

    print(f"\n{search_timing}")

## Generate a grounded answer

The model receives only the retrieved passages, must use numbered evidence markers, and returns the cited sources as clickable links. Generation is disabled during `Run All` unless `AUTHOR_CORPUS_RUN_GROUNDED_ANSWER=1`. Continue to use the exact catalog—not this semantic answerer—for totals and exhaustive lists.

In [ ]:
from author_corpus import GroundedAnswerer
from author_corpus.local_llm import LocalModelSettings, OpenAICompatibleCompleter

local_model_settings = None
grounded_answerer = None
if run_options.run_grounded_answer and selected_search is not None:
    model_id = os.getenv("OLLAMA_MODEL")
    if not model_id or model_id == "your-local-chat-model":
        raise RuntimeError("Set OLLAMA_MODEL to an installed local chat model.")
    local_model_settings = LocalModelSettings(
        model_id=model_id,
        base_url=os.getenv("OLLAMA_BASE_URL", "http://localhost:11434/v1"),
        reasoning_effort=os.getenv("OLLAMA_REASONING_EFFORT", "none"),
    )
    grounded_answerer = GroundedAnswerer(
        selected_search,
        OpenAICompatibleCompleter(local_model_settings),
        model_id=local_model_settings.model_id,
        evidence_limit=selected_search.default_limit,
    )
elif route_decision.route is QueryRoute.EXACT_CATALOG:
    print("Grounded generation skipped because this question belongs to the exact catalog.")
else:
    print("Grounded generation skipped; semantic evidence above is still available for inspection.")

In [ ]:
from author_corpus import GenerationTraceSettings, QueryTrace, QueryTraceStore, RetrievalTraceSettings

grounded_answer = None
query_trace = None
if grounded_answerer is None or local_model_settings is None:
    print("No model call or query trace was created.")
else:
    answer_started = timings.start()
    grounded_answer = grounded_answerer.answer_from_search_result(
        semantic_result,
        question=author_resolution.grounding_question,
    )
    answer_timing = timings.finish("Grounded answer generation", answer_started)
    display(Markdown(grounded_answer.to_markdown()))
    print(answer_timing)

    trace_started = timings.start()
    query_trace = QueryTrace.from_grounded_answer(
        grounded_answer,
        corpus_fingerprint=fingerprint,
        document_content_hashes={document.document_id: document.content_hash for document in documents},
        retrieval=RetrievalTraceSettings(
            evidence_limit=grounded_answerer.evidence_limit,
            max_passages_per_document=selected_search.max_passages_per_document,
            minimum_document_author_fraction=minimum_document_author_fraction,
            strategy=selected_search.strategy,
            score_kind=selected_search.score_kind,
        ),
        generation=GenerationTraceSettings.from_local_settings(
            local_model_settings,
            prompt_version=grounded_answer.prompt_version,
        ),
        elapsed_seconds=answer_timing.elapsed_seconds,
        user_query=semantic_query,
        author_scope=default_scope,
    )
    QueryTraceStore(layout.query_trace_path).put(query_trace)
    print("query trace:", query_trace.trace_id)
    covered_spans, cited_passages = query_trace.cited_span_coverage
    print(f"exact cited evidence spans: {covered_spans}/{cited_passages}")
    print(timings.finish("Persist query trace", trace_started))

## Run routing and retrieval evaluations

The source-agnostic routing benchmark reports precision, recall, exact-tool accuracy, and mistakes. Corpus-specific retrieval judgments remain in an ignored YAML file; dense, lexical, and hybrid strategies are measured separately.

In [ ]:
from author_corpus import (
    evaluate_claim_classifier,
    evaluate_generated_claims,
    evaluate_query_router,
    evaluate_retrieval_strategies,
    load_claim_classification_cases,
    load_generated_claim_cases,
    load_retrieval_cases,
    load_routing_cases,
    validate_claim_classification_sources,
)

routing_evaluation_value = os.getenv(
    "AUTHOR_CORPUS_ROUTING_EVAL",
    str(project_root / "evaluation" / "query-routing.yaml"),
)
routing_evaluation_path = Path(routing_evaluation_value).expanduser()
if not routing_evaluation_path.is_absolute():
    routing_evaluation_path = project_root / routing_evaluation_path
routing_evaluation_started = timings.start()
routing_evaluation = evaluate_query_router(load_routing_cases(routing_evaluation_path.resolve()))
print("routing cases:", len(routing_evaluation.cases))
print("route accuracy:", f"{routing_evaluation.accuracy:.1%}")
exact_tool_accuracy = routing_evaluation.exact_tool_accuracy
exact_tool_metric = "<no exact labels>" if exact_tool_accuracy is None else f"{exact_tool_accuracy:.1%}"
print("exact tool accuracy:", exact_tool_metric)
for metrics in routing_evaluation.route_metrics:
    print(f"{metrics.route}: support={metrics.support}, precision={metrics.precision:.1%}, recall={metrics.recall:.1%}")
for case in routing_evaluation.cases:
    if not case.catalog_tool_correct:
        print("routing miss:", case.name, "expected", case.expected_route, "got", case.predicted_route)
print(timings.finish("Query-routing evaluation", routing_evaluation_started))

evaluation_value = os.getenv("AUTHOR_CORPUS_RETRIEVAL_EVAL")
if not evaluation_value:
    print("Set AUTHOR_CORPUS_RETRIEVAL_EVAL to run local retrieval cases.")
else:
    evaluation_path = Path(evaluation_value).expanduser()
    if not evaluation_path.is_absolute():
        evaluation_path = project_root / evaluation_path
    evaluation_cases = load_retrieval_cases(evaluation_path.resolve())
    evaluation_started = timings.start()
    benchmark = evaluate_retrieval_strategies(
        retrieval_profiles.evaluation_searches(),
        evaluation_cases,
        top_k=6,
        query_transform=lambda query: resolve_author_query(query, author_identity).retrieval_query,
    )
    print("cases:", len(evaluation_cases))
    for strategy_result in benchmark.strategies:
        evaluation = strategy_result.evaluation
        document_metrics = (
            f"document hit={evaluation.hit_rate:.1%}, MRR={evaluation.mean_reciprocal_rank:.3f}, "
            f"nDCG={evaluation.mean_normalized_discounted_cumulative_gain:.3f}"
        )
        if evaluation.passage_hit_rate is None:
            passage_metrics = "passage metrics=<no passage labels>"
        else:
            passage_metrics = (
                f"passage hit={evaluation.passage_hit_rate:.1%}, "
                f"MRR={evaluation.passage_mean_reciprocal_rank:.3f}, "
                f"nDCG={evaluation.passage_mean_normalized_discounted_cumulative_gain:.3f}"
            )
        print(
            f"{strategy_result.strategy}: {document_metrics}; {passage_metrics}; elapsed={strategy_result.elapsed_seconds:.3f}s"
        )
    hybrid_evaluation = next(item.evaluation for item in benchmark.strategies if item.strategy == "hybrid_rrf")
    for case in hybrid_evaluation.cases:
        message = f"{case.name}: first document rank = {case.first_relevant_rank}"
        if case.has_passage_expectations:
            message += f"; first passage rank = {case.first_relevant_passage_rank}"
        print(message)
    print(timings.finish("Retrieval evaluation", evaluation_started))

claim_evaluation_value = os.getenv(
    "AUTHOR_CORPUS_CLAIM_EVAL",
    str(project_root / "evaluation" / "claim-classification.yaml"),
)
claim_evaluation_path = Path(claim_evaluation_value).expanduser()
if not claim_evaluation_path.is_absolute():
    claim_evaluation_path = project_root / claim_evaluation_path
claim_evaluation_started = timings.start()
claim_cases = load_claim_classification_cases(claim_evaluation_path.resolve())
claim_source_issues = validate_claim_classification_sources(
    claim_cases,
    {document.document_id: document for document in documents},
)
if claim_source_issues:
    issue_summary = "; ".join(f"{issue.span_id}: {issue.code}" for issue in claim_source_issues)
    raise RuntimeError(f"Claim-classification labels have stale source provenance: {issue_summary}")
claim_evaluation = evaluate_claim_classifier(claim_cases)
print("claim cases:", len(claim_evaluation.cases))
print("claim provenance coverage:", f"{claim_evaluation.provenance_coverage:.1%}")
print("claim accuracy:", f"{claim_evaluation.accuracy:.1%}")
print("claim coverage:", f"{claim_evaluation.coverage:.1%}")
selective_accuracy = claim_evaluation.selective_accuracy
selective_metric = "<no resolved cases>" if selective_accuracy is None else f"{selective_accuracy:.1%}"
print("claim selective accuracy:", selective_metric)
false_support_rate = claim_evaluation.false_support_rate
false_support_metric = "<no non-support labels>" if false_support_rate is None else f"{false_support_rate:.1%}"
print("claim false-support rate:", false_support_metric)
for metrics in claim_evaluation.label_metrics:
    print(f"{metrics.label}: support={metrics.support}, precision={metrics.precision:.1%}, recall={metrics.recall:.1%}")
for case in claim_evaluation.mistakes:
    print(
        "claim-classification miss:",
        case.name,
        "expected",
        case.expected_label,
        "got",
        case.decision.label,
    )
print(timings.finish("Claim-classification evaluation", claim_evaluation_started))

generated_claim_evaluation_value = os.getenv("AUTHOR_CORPUS_GENERATED_CLAIM_EVAL")
if not generated_claim_evaluation_value:
    print("Set AUTHOR_CORPUS_GENERATED_CLAIM_EVAL to evaluate private trace-derived claim labels.")
else:
    generated_claim_path = Path(generated_claim_evaluation_value).expanduser()
    if not generated_claim_path.is_absolute():
        generated_claim_path = project_root / generated_claim_path
    generated_claim_started = timings.start()
    generated_claim_evaluation = evaluate_generated_claims(
        load_generated_claim_cases(generated_claim_path.resolve()),
        QueryTraceStore(layout.query_trace_path),
        documents={document.document_id: document for document in documents},
    )
    print("generated claim cases:", len(generated_claim_evaluation.cases))
    print("generated claim accuracy:", f"{generated_claim_evaluation.accuracy:.1%}")
    print("generated claim exact-span coverage:", f"{generated_claim_evaluation.exact_span_coverage:.1%}")
    generated_false_support = generated_claim_evaluation.false_support_rate
    generated_false_support_metric = (
        "<no non-support labels>" if generated_false_support is None else f"{generated_false_support:.1%}"
    )
    print("generated claim false-support rate:", generated_false_support_metric)
    print("generated claim abstention rate:", f"{generated_claim_evaluation.abstention_rate:.1%}")
    print(timings.finish("Generated-claim evaluation", generated_claim_started))

## Prepare human claim-review proposals

A classifier result is only a suggestion. This cell prepares source-bound proposals and reads recent review records, but it does not accept, revise, reject, persist, or apply anything. After inspecting a proposal and its exact evidence, call `review_claim_proposal` with an identified reviewer and an explicit action, persist the returned record with `ClaimReviewStore.put`, and apply only accepted or revised records to an `EvidenceLedger` with `apply_claim_review`.

In [ ]:
from author_corpus import ClaimReviewProposal, ClaimReviewStore

review_preparation_started = timings.start()
review_proposals = tuple(
    ClaimReviewProposal.from_decision(
        result.decision,
        corpus_fingerprint=fingerprint,
        claim_id=f"review-{case.name}",
        evidence_spans=(case.evidence_span,),
    )
    for case, result in zip(claim_cases, claim_evaluation.cases, strict=True)
    if case.evidence_span is not None
)
review_store = ClaimReviewStore(layout.claim_review_path)
recent_reviews = review_store.recent(limit=5, corpus_fingerprint=fingerprint)
print("source-bound proposals awaiting human action:", len(review_proposals))
print("recent persisted human reviews:", len(recent_reviews))
for proposal in review_proposals[:5]:
    print(
        proposal.claim_id,
        "classifier=",
        proposal.classifier_decision.label,
        "suggested status=",
        proposal.suggested_status or "<revision required>",
    )
print("No review action was taken and no AuditedClaim was created.")
print(timings.finish("Prepare human claim-review proposals", review_preparation_started))

## Inspect cached navigation summaries

This notebook only reports cache coverage. It never resumes summary generation or corpus-wide synthesis during `Run All`. Build resumable per-document navigation summaries with `uv run --extra local author-corpus build-knowledge`. The unaudited broad synthesis remains available only through the explicit `--include-experimental-synthesis` flag.

In [ ]:
from author_corpus import SummaryStore
from author_corpus.summaries import DOCUMENT_SUMMARY_PROMPT_VERSION

knowledge_store = SummaryStore(layout.summary_path)
knowledge_model_id = os.getenv("OLLAMA_MODEL", "").strip()
if not knowledge_model_id or knowledge_model_id == "your-local-chat-model":
    print("Set OLLAMA_MODEL to inspect model-specific navigation-summary coverage.")
else:
    status_started = timings.start()
    missing_summaries = knowledge_store.missing(
        documents,
        model_id=knowledge_model_id,
        prompt_version=DOCUMENT_SUMMARY_PROMPT_VERSION,
    )
    print("cached navigation summaries:", len(documents) - len(missing_summaries))
    print("missing navigation summaries:", len(missing_summaries))
    print("unaudited broad synthesis: paused")
    print(timings.finish("Inspect navigation-summary cache", status_started))

In [ ]:
from author_corpus import EvidenceSynthesisStore

synthesis_inspection_started = timings.start()
evidence_synthesis_store = EvidenceSynthesisStore(layout.evidence_synthesis_path)
evidence_synthesis = evidence_synthesis_store.latest(
    corpus_fingerprint=fingerprint,
    author_scope=default_scope,
)
if evidence_synthesis is None:
    print("No evidence-bound synthesis exists for the current corpus and author scope.")
else:
    synthesis_issues = evidence_synthesis.validate_sources({document.document_id: document for document in documents})
    print("evidence-bound synthesis claims:", len(evidence_synthesis.claims))
    print("evidence-bound synthesis source issues:", len(synthesis_issues))
    print("evidence-bound synthesis coverage:", evidence_synthesis.coverage)
print(timings.finish("Inspect evidence-bound synthesis", synthesis_inspection_started))

## Timing report

Compare cold-build and warm-cache timings before changing models, chunking, or persistence behavior.

In [ ]:
print(timings.format_report())

## Milestone status and next work

Completed: source-agnostic ingestion, exact catalog execution, structure- and voice-aware indexing, measured dense/BM25 retrieval, deterministic query routing, conservative configured author identity, exact catalog-name and explicit UI/API multi-author scopes, hard author filtering across dense/BM25/fused retrieval, comparison coverage abstention, bounded conversation, exact citation spans, fail-closed citation repair with durable private attempt diagnostics, durable reasoning-aware traces, offline configured and generated-claim evaluation, opt-in two-round claim-level reasoning, conservative and experimental structured semantic claim verifiers, private aggregate verifier comparison with safety metrics and timing, explicit claim and whole-answer review, private evidence-in training export, evidence-bound ledger synthesis, and the local Gradio chat/review interface.

Next:

1. Grow the initial human-labeled private routing regression set with unseen queries. Treat tuned examples as development cases rather than proof of generalization.
2. Grow the private author-identity retrieval cases, including aliases and questions that should abstain, without adding fuzzy nickname inference.
3. Grow provenance-checked configured and trace-derived claim/evidence development sets while preserving unseen query/source groups as holdouts.
4. Populate separate private development and unseen holdout sets for `evaluate-verifiers`; do not change the conservative default until the semantic verifier preserves an acceptable false-acceptance rate across authors, sites, document types, quotations, and coauthored work.
5. Accumulate explicitly reviewed claims and answers, then populate evidence-bound synthesis without treating navigation summaries as facts.
6. Grow synthetic and private multi-author retrieval holdouts. Add a per-author alias registry only when a second author needs aliases; keep fuzzy identity inference disabled.
7. Consider an optional LoRA/QLoRA only after reviewed examples are numerous enough for a meaningful held-out comparison.
8. Build the pinned public-domain multi-author corpus and static evidence explorer described in `docs/public-demo.md`; add Pages deployment only after the public assets and regression report are reproducible.

A dedicated biographical query route is deferred until real query labels show that it is distinct enough from broad discovery to justify another execution path.